### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="bad_customer_detection",
    dataset_year="2020",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/podsyp/is-this-a-good-customer",
    download_description="""
Download the dataset from Kaggle using the Kaggle API. 

mkdir -p local-data-warehouse/bad_customer_detection/ && cd local-data-warehouse/bad_customer_detection && kaggle datasets download podsyp/is-this-a-good-customer && unzip is-this-a-good-customer.zip && rm is-this-a-good-customer.zip && cd ../../
""",
    # References
    academic_reference_bibtex="""@misc{Podsyp2020IsThisAGoodCustomer,
  title={Is This a Good Customer?},
  author={Podsyp},
  year={2020},
  publisher={Kaggle},
  url={https://www.kaggle.com/datasets/podsyp/is-this-a-good-customer}
}
""",
    academic_reference_bibtex_key="Podsyp2020IsThisAGoodCustomer",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We renamed the values of the target variable to be more descriptive.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="bad_customer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="bad_customer",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "clients.csv")

target_feature = "bad_customer"
df.rename(columns={"bad_client_target": target_feature}, inplace=True)
df[target_feature] = df[target_feature].map({0: "No", 1: "Yes"}).astype("category")

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

cat_cols = ['sex', 'education', 'product_type', 'having_children_flg', 'region', 'family_status', 'phone_operator', 'is_client',]
df[cat_cols] = df[cat_cols].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()